# Lab 1-05：雙軌客戶端——本地掛了自動切雲端

**目標**：寫一個小客戶端，平常走本機 Ollama（免費、資料不出門），本機連不上就自動改走雲端。
寫完的成品就是 `course_utils/llm.py`，之後每個模組都會用它。

**你會學到**
- 用環境變數管理端點與金鑰，不寫死在程式裡
- 最簡單、也最實用的容錯：依序試，失敗就換下一個
- 串流版本怎麼做同樣的容錯

**前置**：Lab 1-04 能跑通。雲端備援是選填：在 `.env` 填 `CLOUD_API_KEY` 才會啟用。

**對應投影片**：「Lab 1 雙軌客戶端架構」「Lab 1 串流程式片段」

## Step 1：端點從環境變數來

金鑰和網址放在 `.env`，程式只負責讀。這樣同一份程式碼，換台機器、換個服務都不用改。

In [1]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Endpoint:
    name: str
    base_url: str
    model: str
    api_key: str


def endpoints() -> list[Endpoint]:
    eps = [Endpoint(
        name="local",
        base_url=os.getenv("LOCAL_BASE_URL", "http://localhost:11434/v1"),
        model=os.getenv("LOCAL_MODEL", "qwen2.5:3b"),
        api_key="ollama",
    )]
    if os.getenv("CLOUD_API_KEY"):
        eps.append(Endpoint(
            name="cloud",
            base_url=os.getenv("CLOUD_BASE_URL", "https://api.deepseek.com/v1"),
            model=os.getenv("CLOUD_MODEL", "deepseek-chat"),
            api_key=os.environ["CLOUD_API_KEY"],
        ))
    return eps


for ep in endpoints():
    print(ep.name, ep.base_url, ep.model)  # 不印金鑰

local http://localhost:11434/v1 qwen2.5:3b


## Step 2：對單一端點送一次請求

In [2]:
from openai import OpenAI, OpenAIError


def ask(ep: Endpoint, messages: list[dict]) -> str:
    client = OpenAI(base_url=ep.base_url, api_key=ep.api_key, timeout=30, max_retries=0)
    r = client.chat.completions.create(model=ep.model, messages=messages, temperature=0)
    return r.choices[0].message.content


print(ask(endpoints()[0], [{"role": "user", "content": "用一句話介紹娜美。"}]))

娜美是《海贼王》中的海贼船“蒙德利安”的船长，以其出色的航海技能和坚定的意志力著称。


## Step 3：依序試，失敗就換下一個

容錯不用複雜：把端點排好優先順序，一個一個試，全部失敗才報錯，並把每個端點的錯誤都列出來。

In [3]:
def chat(messages: list[dict]) -> str:
    errors = []
    for ep in endpoints():
        try:
            answer = ask(ep, messages)
            print(f"[用了 {ep.name}]")
            return answer
        except OpenAIError as exc:
            errors.append(f"{ep.name}: {type(exc).__name__}: {exc}")
    raise RuntimeError("所有端點都失敗：\n" + "\n".join(errors))


print(chat([{"role": "user", "content": "用一句話介紹騙人布。"}]))

[用了 local]
騙人布是一種設計精巧的布料，其表面看起來與真絲相似，但實際上是由化學纖維制成，用以騙取消費者信任和資金。


## Step 4：故意把本地弄壞

把本地端點改成一個沒人在聽的 port，模擬 Ollama 當掉。
- 有填 `CLOUD_API_KEY`：會看到 `[用了 cloud]`
- 沒填：會看到清楚的錯誤訊息，告訴你每個端點各自錯在哪

In [4]:
original = os.environ.get("LOCAL_BASE_URL")
os.environ["LOCAL_BASE_URL"] = "http://localhost:9/v1"
try:
    print(chat([{"role": "user", "content": "用一句話介紹香吉士。"}]))
except RuntimeError as exc:
    print(exc)
finally:  # 改回原本的設定
    if original is None:
        del os.environ["LOCAL_BASE_URL"]
    else:
        os.environ["LOCAL_BASE_URL"] = original

所有端點都失敗：
local: APIConnectionError: Connection error.


## Step 5：串流版

串流的容錯有個細節：**只有還沒吐出任何字之前才能切換**。已經印了半句再換端點，使用者會看到兩段拼在一起的回答。
這裡的做法是：連線和第一個片段都成功，才算這個端點可用。

In [5]:
def stream(messages: list[dict]):
    errors = []
    for ep in endpoints():
        started = False
        try:
            client = OpenAI(base_url=ep.base_url, api_key=ep.api_key, timeout=30, max_retries=0)
            chunks = client.chat.completions.create(
                model=ep.model, messages=messages, temperature=0, stream=True
            )
            for chunk in chunks:
                if chunk.choices and chunk.choices[0].delta.content:
                    started = True
                    yield chunk.choices[0].delta.content
            return
        except OpenAIError as exc:
            if started:  # 已經吐出字了，不能再換端點
                raise
            errors.append(f"{ep.name}: {exc}")
    raise RuntimeError("所有端點都失敗：\n" + "\n".join(errors))


for piece in stream([{"role": "user", "content": "用三句話說明為什麼魯夫戴草帽。"}]):
    print(piece, end="", flush=True)

魯夫戴草帽主要是因為他想要隱藏自己的真實身份，避免被他人認出。這個習慣也成為了他獨特個性的一部分，象征著他的自由和不羈。此外，草帽還代表了海洋和自由，與魯夫的航海家身份相契合。

## Step 6：這就是 `course_utils/llm.py`

把上面的函式整理一下、加上 embedding，就是課程共用的客戶端。之後的 notebook 都這樣用：

In [6]:
from course_utils.llm import chat as course_chat, stream as course_stream

print(course_chat([{"role": "user", "content": "一句話：草帽海賊團的船醫是誰？"}]))
for piece in course_stream([{"role": "user", "content": "一句話：草帽海賊團的劍士是誰？"}]):
    print(piece, end="", flush=True)

草帽海賊團的船醫是莫德雷德。
草帽海賊團的劍士是山賊（山治）。

## 試試看

1. 在 `chat` 裡加上簡單的重試：同一個端點失敗時等 1 秒再試一次，第二次還失敗才換下一個。
2. 每次呼叫把「用了哪個端點、花了幾秒」記到一個 list，跑十次後印出平均延遲。
3. 想一想：哪些資料不能送到雲端？如果有這種資料，`chat` 應該怎麼改？

## 小結

- 端點和金鑰放環境變數；程式只讀，不寫死。
- 容錯＝依序試、失敗換下一個、全部失敗時把每個錯誤都講清楚。
- 串流只能在第一個字之前切換。模組 1 到這裡結束；模組 2 開始教怎麼把 prompt 寫好、讓輸出變成程式能用的格式。